# Datamine IPDNPG Process Example

This notebook demonstrates how to configure and run the **`ipdnpg`** process wrapper in `dmstudio`.

## Process Description

## IPDNPG

Interpolate grades into block model cells using non-parametric methods.

### Input Files:

* **proto** (Block Model Prototype):
Prototype model. Must contain at least the fields XC, YC, ZC, XINC, YINC, ZINC, XMORIG,
YMORIG, ZMORIG, NX, NY, NZ, IJK. May contain cells and sub-cells.
Required=Yes

* **in** (Undefined):
Input sample data (sorted on X). Must contain the fields X , Y , Z , VALUE.
Required=Yes

### Output Files:

* **model** (Block Model):
Output interpolated model.
Required=Yes

### Fields:

* **x** (Numeric : IN):
Name of sample X field.
Default=X
Required=Yes

* **y** (Numeric : IN):
Name of sample Y field.
Default=Y
Required=Yes

* **z** (Numeric : IN):
Name of sample Z field.
Default=Z
Required=Yes

* **value** (Numeric : IN):
Name of field to be interpolated.
Default=Undefined
Required=Yes

### Parameters:

* **type**:
(1) Type of interpolation to be used :- Options: 1: Moving mode; 2: Moving median; 3:
Varying quantile; 4: Angle-weighted varying quantile
Range=1,4
Values=1,2,3,4
Default=1
Required=Yes

* **radius**:
Search radius [mean cell dimension].
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **minnop**:
Minimum number of samples (5).
Range=Undefined
Values=Undefined
Default=5
Required=No

* **maxnop**:
Maximum number of samples (1000).
Range=Undefined
Values=Undefined
Default=1000
Required=No

* **xsubcell**:
No. of sub-cells/cell in X (1).
Range=Undefined
Values=Undefined
Default=1
Required=No

* **ysubcell**:
No. of sub-cells/cell in Y (1).
Range=Undefined
Values=Undefined
Default=1
Required=No

* **zsubcell**:
No. of sub-cells/cell in Z (1). Above three parameters only used if input prototype
does not already contain cells.
Range=Undefined
Values=Undefined
Default=1
Required=No

* **print**:
>=2 Display co-ordinates and interpolated values.
Range=0,2
Values=0,2
Default=0
Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('ipdnpg')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute ipdnpg
print("Running ipdnpg...")
dm_cmd.ipdnpg(
    proto_i='t_mod1',  # required
    in_i='t_assays',  # required
    model_o='t_ipdnpg_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    z_f='Z',  # required
    value_f='optional',  # required
    type_p='required_val',  # required
    # radius_p='optional',  # optional
    # minnop_p=5,  # optional
    # maxnop_p=1000,  # optional
    # xsubcell_p=1,  # optional
    # ysubcell_p=1,  # optional
    # zsubcell_p=1,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("ipdnpg execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_ipdnpg_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")